# SafeVision · Notebook 6 · Explaining the accident-type classifier (XAI, experiment E10)

**Goal:** show *why* the accident-type classifier (Notebook 3) predicts what it predicts, and pick the explanation method that is **most faithful** to the model, instead of choosing one by habit.

For one text, every method gives each word a score: how much that word pushes the model towards the predicted accident type. We compare three popular methods:

| Method | Idea | Reference |
|---|---|---|
| **SHAP** (Partition explainer) | fair share of the prediction for each word (game theory, Shapley values) | Lundberg & Lee 2017 [X1] |
| **LIME** | removes random words and fits a small linear model around this one text | Ribeiro et al. 2016 [X2] |
| **Integrated Gradients (IG)** | follows the gradient from an empty text to the real text | Sundararajan et al. 2017 [X3] |

**How do we know which explanation is right?** We use the faithfulness tests of the ERASER benchmark [X4]:
- **Comprehensiveness**: delete the top-*k* words. If they really mattered, the model's confidence should **drop a lot** (higher is better).
- **Sufficiency**: keep **only** the top-*k* words. If they are enough, the confidence should **stay** (the drop should be small, lower is better).
- A **random** word ranking is the floor every method must beat.

| ID | Question | Main scores |
|---|---|---|
| **E10a** | Which method is most faithful? | comprehensiveness and sufficiency at k = 1, 3, 5 (+ random baseline, Wilcoxon test) |
| **E10b** | Do the methods agree with each other? | Spearman rank correlation, top-3 word overlap |
| **E10c** | Is it fast enough for the app? | milliseconds per explanation |
| **E10d** | What does the model rely on overall? | top words per accident type (global importance) |
| **E10e** | What does it look like for SafeVision situations? | coloured word examples |

The winner is saved as `xai_choice.json` and used by the **Incident Analysis** page of the web app.

**Runtime:** GPU (Kaggle or Colab), about 30-60 min for 200 reports. Progress is saved, so a stopped run continues where it stopped.

## 0. Install libraries
`shap`, `lime` and `captum` (IG for PyTorch), plus the libraries Notebook 3 used.

In [ ]:
# [colab-only]
!nvidia-smi
%pip install -q shap lime captum transformers scikit-learn openpyxl huggingface_hub scipy

## 1. Configuration (edit this cell only)

In [ ]:
# CONFIG
from pathlib import Path
import re, json, random, time, os, importlib.util

ON_KAGGLE = Path("/kaggle/input").exists()
IN_COLAB = importlib.util.find_spec("google.colab") is not None
BASE = Path("/kaggle/working") if ON_KAGGLE else Path("/content") if IN_COLAB else Path.home() / "safevision"
OUT = BASE / "safevision_outputs_nb6"
ART = BASE / "artifacts"
CLASSIFIER_DIR = ART / "accident_classifier"          # the app model saved by Notebook 3
SIR_PAGE = "https://www.osha.gov/severe-injury-reports"
SIR_LOCAL = BASE / "sir_data"

# Same year split as Notebook 3: only the TEST years (never seen in training) are explained.
TRAIN_MAX_YEAR, VAL_YEAR = 2021, 2022
OVERSAMPLE_MIN = 300
SEED = 42

TASKS = ["event_group"]       # outputs to explain: accident type (main). Add "body_group", "severity" if time allows.
METHODS = ["shap", "lime", "ig"]
N_EVAL = 200                  # test reports for E10a-c (more = more precise, slower)
N_GLOBAL = 400                # test reports for the global word importance (E10d)
KS = (1, 3, 5)                # top-k words for comprehensiveness / sufficiency
SHAP_EVALS = 500              # model calls per SHAP explanation
LIME_SAMPLES = 500            # perturbed texts per LIME explanation
IG_STEPS = 50                 # integration steps for IG
MIN_WORDS = 5                 # skip very short texts
MIN_WORD_COUNT = 5            # a word must appear in >= this many reports to enter the global top list

RES, FIGS = OUT / "results", OUT / "figures"
PROGRESS = RES / "xai_progress.jsonl"
HF_REPO, token = None, None   # filled in by the next cells
for p in [OUT, RES, FIGS, ART, SIR_LOCAL]:
    p.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

def get_secret(name):
    # Environment variable, Colab Secrets or Kaggle Secrets (None if not set). Keys are never printed.
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

## 2. Results logger
Same format as the other notebooks, so `merge_results.py` can combine everything.

In [ ]:
import csv, datetime
import pandas as pd
LOG_CSV = RES / "experiments_log.csv"
LOG_FIELDS = ["experiment", "run", "method", "task", "dataset", "metric", "value", "notes", "timestamp"]

def log_result(experiment, run, method, task, metric, value, notes="", dataset="OSHA-SIR construction test years"):
    new_file = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new_file:
            w.writeheader()
        w.writerow(dict(experiment=experiment, run=run, method=method, task=task, dataset=dataset, metric=metric,
                        value=round(float(value), 4), notes=notes, timestamp=datetime.datetime.now().isoformat(timespec="seconds")))

def read_log():
    return pd.read_csv(LOG_CSV) if LOG_CSV.exists() else pd.DataFrame(columns=LOG_FIELDS)

## 3. Get the accident classifier from Notebook 3

Downloaded from your private Hugging Face repo (`nb3_accident_text/models/accident_classifier`). On Kaggle it can also be attached as an input dataset. In Colab you can upload `safevision_nb3_outputs.zip` if the download fails.

In [ ]:
# [colab-only]
import zipfile, shutil, io
from huggingface_hub import login, snapshot_download, HfApi, hf_hub_download
token = get_secret("HF_TOKEN")
HF_REPO = None
if token:
    login(token=token)
    HF_REPO = f"{HfApi(token=token).whoami()['name']}/safevision-artifacts"

def place(root):
    hits = [p.parent for p in Path(root).glob("**/accident_classifier/labels.json")]
    if hits and not CLASSIFIER_DIR.exists():
        shutil.copytree(hits[0], CLASSIFIER_DIR)

if ON_KAGGLE:
    place("/kaggle/input")
if not CLASSIFIER_DIR.exists() and HF_REPO:
    try:
        place(snapshot_download(HF_REPO, token=token, allow_patterns=["nb3_accident_text/**/accident_classifier/**"]))
    except Exception as e:
        print("HF download failed:", e)
while not CLASSIFIER_DIR.exists() and IN_COLAB:
    from google.colab import files
    print("Upload safevision_nb3_outputs.zip:")
    up = files.upload()
    if not up:
        break
    for name, data in up.items():
        zipfile.ZipFile(io.BytesIO(data)).extractall(ART / Path(name).stem)
    place(ART)
assert CLASSIFIER_DIR.exists(), "accident_classifier not found (HF repo, Kaggle input or upload)"
# Continue an interrupted run
try:
    if HF_REPO and not PROGRESS.exists():
        shutil.copy(hf_hub_download(HF_REPO, "nb6_progress/xai_progress.jsonl", token=token), PROGRESS)
        print("Found saved progress:", sum(1 for _ in open(PROGRESS)), "explanations will be reused.")
except Exception:
    print("No saved progress yet (normal on the first run).")

## 4. Shared SafeVision modules (same files as the web-app backend)
- `classifier.py`: the accident-type classifier (Notebook 3), with `proba()` for the explainers,
- `explain.py`: SHAP, LIME and IG in one common format + the faithfulness tests.

In [ ]:
# ---- backend/app/classifier.py ----
"""SafeVision accident-type classifier (N2): loads the context-only multi-task DistilBERT saved by Notebook 3.

Predicts the likely accident type (OIICS event group), injured body-part group and severity for a
situation description, following Wang et al. 2025 [6] (OIICS groups, DistilBERT) and Cao et al. 2026 [7].
"""
import json, re
from pathlib import Path


class AccidentClassifier:
    """Multi-task text classifier: shared encoder + one linear head per output."""

    def __init__(self, model_dir, device="cpu"):
        import torch, torch.nn as nn
        from transformers import AutoModel, AutoTokenizer
        d = Path(model_dir)
        meta = json.load(open(d / "labels.json"))
        self.targets, self.max_len, self.device = meta["targets"], meta["max_len"], device
        self.out_re = re.compile(rf"\b({meta['outcome_words']})\b", re.I)
        self.mech_re = re.compile(rf"\b({meta['mechanism_words']})\b", re.I)
        self.tok = AutoTokenizer.from_pretrained(d / "encoder")
        self.encoder = AutoModel.from_pretrained(d / "encoder").to(device).eval()
        hidden = self.encoder.config.hidden_size
        self.heads = nn.ModuleDict({t: nn.Linear(hidden, len(ls)) for t, ls in self.targets.items()})
        self.heads.load_state_dict(torch.load(d / "heads.pt", map_location=device, weights_only=True))
        self.heads.to(device).eval()
        self._torch = torch

    def context_only(self, text):
        """Same masking as in training: remove outcome and mechanism words."""
        t = self.mech_re.sub(" ", self.out_re.sub(" ", text))
        return re.sub(r"\s+", " ", t).strip()

    def predict(self, text, top=3):
        """Top classes with probabilities for each output, e.g. {'event_group': [('Fall, slip or trip', 0.41), ...]}."""
        torch = self._torch
        enc = self.tok([self.context_only(text)], truncation=True, max_length=self.max_len, return_tensors="pt").to(self.device)
        with torch.no_grad():
            h = self.encoder(**{k: v for k, v in enc.items() if k in ("input_ids", "attention_mask")}).last_hidden_state[:, 0]
            out = {}
            for t, head in self.heads.items():
                p = head(h).softmax(-1)[0].cpu().numpy()
                order = p.argsort()[::-1][:top]
                out[t] = [(self.targets[t][i], round(float(p[i]), 3)) for i in order]
        return out

    def proba(self, texts, task="event_group", mask=False, batch=64):
        """Class probabilities (n_texts x n_classes) for one output. Used by the explainers (SHAP, LIME, IG).

        mask=False: the texts are used as given (the explainers already work on the context-only text)."""
        import numpy as np
        torch = self._torch
        texts = [self.context_only(t) if mask else t for t in texts]
        out = []
        for i in range(0, len(texts), batch):
            chunk = [t if t.strip() else "[UNK]" for t in texts[i:i + batch]]     # an empty text still gets a score
            enc = self.tok(chunk, truncation=True, max_length=self.max_len, padding=True, return_tensors="pt").to(self.device)
            with torch.no_grad():
                h = self.encoder(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state[:, 0]
                out.append(self.heads[task](h).softmax(-1).cpu().numpy())
        return np.concatenate(out) if out else np.zeros((0, len(self.targets[task])))

In [ ]:
# ---- backend/app/explain.py ----
"""SafeVision word explanations for the accident-type classifier (XAI, experiment E10).

Three methods give every word of the (context-only) text a score: how much it pushes the model towards the
predicted class.
  * SHAP  - Partition explainer with a text masker (Lundberg & Lee 2017; shap library).
  * LIME  - local linear model on randomly removed words (Ribeiro et al. 2016; lime library).
  * IG    - Integrated Gradients on the word embeddings (Sundararajan et al. 2017; Captum).
All three return the same format, so they can be compared with the same faithfulness tests (ERASER,
DeYoung et al. 2020): comprehensiveness (remove the top words: the score should drop a lot) and
sufficiency (keep only the top words: the score should stay high, so the drop should be small).
"""
import json, re, time
from pathlib import Path
import numpy as np

ALL_METHODS = ("shap", "lime", "ig")


def words_of(text):
    return text.split()


class WordExplainer:
    def __init__(self, classifier, shap_evals=500, lime_samples=500, ig_steps=50, seed=0):
        self.clf, self.seed = classifier, seed
        self.shap_evals, self.lime_samples, self.ig_steps = shap_evals, lime_samples, ig_steps
        self._shap, self._lime, self._ig = {}, None, None

    # ---------- shared helpers ----------
    def prob_fn(self, task, target):
        """f(list of texts) -> probability of class `target` (index) for each text."""
        return lambda texts: self.clf.proba(list(texts), task)[:, target]

    def predict_index(self, text, task):
        return int(self.clf.proba([text], task)[0].argmax())

    # ---------- SHAP ----------
    def shap_scores(self, text, task, target):
        import shap
        if task not in self._shap:
            # Hidden words are replaced by a marker that is then deleted, so SHAP, LIME and the faithfulness
            # tests all "remove" words the same way.
            masker = shap.maskers.Text(r"\s+", mask_token="XXDELXX", collapse_mask_token=True)
            drop = lambda t: re.sub(r"\s+", " ", t.replace("XXDELXX", " ")).strip()
            f = lambda texts: self.clf.proba([drop(t) for t in texts], task)
            self._shap[task] = shap.Explainer(f, masker, output_names=self.clf.targets[task], algorithm="partition")
        ex = self._shap[task]([text], max_evals=self.shap_evals, silent=True)
        toks, vals = ex.data[0], ex.values[0][:, target]
        scores = [float(v) for t, v in zip(toks, vals) if str(t).strip()]
        return self._fit(scores, len(words_of(text)))

    # ---------- LIME ----------
    def lime_scores(self, text, task, target):
        from lime.lime_text import LimeTextExplainer
        if self._lime is None:
            self._lime = LimeTextExplainer(split_expression=r"\s+", bow=False, mask_string="", random_state=self.seed)
        f = lambda texts: self.clf.proba(list(texts), task)
        n = len(words_of(text))
        ex = self._lime.explain_instance(text, f, labels=[target], num_features=max(n, 1), num_samples=self.lime_samples)
        scores = np.zeros(n)
        for pos, w in ex.as_map()[target]:
            if pos < n:
                scores[pos] = w
        return scores.tolist()

    # ---------- Integrated Gradients ----------
    def ig_scores(self, text, task, target):
        import torch
        from captum.attr import LayerIntegratedGradients
        clf, words = self.clf, words_of(text)
        enc = clf.tok(words, is_split_into_words=True, truncation=True, max_length=clf.max_len, return_tensors="pt")
        ids, att = enc["input_ids"].to(clf.device), enc["attention_mask"].to(clf.device)
        special = torch.tensor([i in clf.tok.all_special_ids for i in ids[0].tolist()], device=clf.device)
        base = torch.where(special, ids, torch.full_like(ids, clf.tok.pad_token_id))   # baseline: every word replaced by [PAD]

        def forward(input_ids, attention_mask):
            h = clf.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
            return clf.heads[task](h).softmax(-1)[:, target]

        lig = LayerIntegratedGradients(forward, clf.encoder.embeddings)
        attr = lig.attribute(ids, baselines=base, additional_forward_args=(att,), n_steps=self.ig_steps,
                             internal_batch_size=self.ig_steps)
        sub = attr.sum(-1)[0].detach().cpu().numpy()          # one number per sub-word token
        scores = np.zeros(len(words))
        for tok_i, w in enumerate(enc.word_ids(0)):
            if w is not None:
                scores[w] += sub[tok_i]                      # sum the sub-words of each word
        return scores.tolist()

    # ---------- public API ----------
    def explain(self, text, task="event_group", method="shap", target=None):
        """Word scores for one text. Returns dict(words, scores, target, label, prob, method, ms)."""
        t0 = time.time()
        target = self.predict_index(text, task) if target is None else target
        fn = {"shap": self.shap_scores, "lime": self.lime_scores, "ig": self.ig_scores}[method]
        scores = fn(text, task, target) if words_of(text) else []
        p = float(self.clf.proba([text], task)[0, target])
        return dict(words=words_of(text), scores=[round(float(s), 5) for s in scores], target=target,
                    label=self.clf.targets[task][target], prob=round(p, 4), method=method,
                    ms=round(1000 * (time.time() - t0), 1))

    @staticmethod
    def _fit(scores, n):
        # Make the score list exactly n long (tokeniser edge cases).
        return (list(scores) + [0.0] * n)[:n]


# ---------- faithfulness (ERASER) ----------
def top_k_positions(scores, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return set(order[:k].tolist())


def remove_words(words, positions):
    return " ".join(w for i, w in enumerate(words) if i not in positions)


def keep_words(words, positions):
    return " ".join(w for i, w in enumerate(words) if i in positions)


def faithfulness(prob_fn, words, scores, ks=(1, 3, 5)):
    """Comprehensiveness and sufficiency at each k. prob_fn(list of texts) -> target-class probabilities."""
    full = " ".join(words)
    texts = [full]
    for k in ks:
        pos = top_k_positions(scores, min(k, len(words)))
        texts += [remove_words(words, pos), keep_words(words, pos)]
    p = prob_fn(texts)
    out = {}
    for j, k in enumerate(ks):
        out[f"comp@{k}"] = float(p[0] - p[1 + 2 * j])
        out[f"suff@{k}"] = float(p[0] - p[2 + 2 * j])
    return out


def load_choice(models_dir):
    """Method picked by Notebook 6 (xai_choice.json), default SHAP."""
    f = Path(models_dir) / "xai_choice.json"
    return json.load(open(f)).get("best_method", "shap") if f.exists() else "shap"

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
clf = AccidentClassifier(CLASSIFIER_DIR, device=DEVICE)
XP = WordExplainer(clf, shap_evals=SHAP_EVALS, lime_samples=LIME_SAMPLES, ig_steps=IG_STEPS, seed=SEED)
meta = json.load(open(CLASSIFIER_DIR / "labels.json"))
OUTCOME_WORDS, MECHANISM_WORDS = meta["outcome_words"], meta["mechanism_words"]   # exactly the masking used in training
print("Device:", DEVICE, "| outputs:", {t: len(v) for t, v in clf.targets.items()})

## 5. Download the OSHA Severe Injury Reports
Same data as Notebook 3. On Kaggle you can also attach the SIR CSV as an input dataset.

In [ ]:
# [colab-only]
import requests, zipfile, io, shutil
kaggle_csv = [p for p in Path("/kaggle/input").rglob("*.csv") if "narrative" in open(p, encoding="latin-1").readline().lower()] if ON_KAGGLE else []
if kaggle_csv:
    shutil.copy(kaggle_csv[0], SIR_LOCAL / "sir.csv")
else:
    try:
        html = requests.get(SIR_PAGE, timeout=60, headers={"User-Agent": "Mozilla/5.0"}).text
        link = re.search(r'href="([^"]+\.zip)"', html).group(1)
        link = link if link.startswith("http") else "https://www.osha.gov" + link
        data = requests.get(link, timeout=300, headers={"User-Agent": "Mozilla/5.0"}).content
    except Exception as e:
        from google.colab import files
        print("Automatic download failed (", e, "). Please upload the SIR ZIP or CSV:")
        data = list(files.upload().values())[0]
    if data[:2] == b"PK":
        zipfile.ZipFile(io.BytesIO(data)).extractall(SIR_LOCAL)
    else:
        (SIR_LOCAL / "sir.csv").write_bytes(data)
print(list(SIR_LOCAL.rglob("*.csv")))

## 6. Rebuild the Notebook 3 test set
These four cells are copied from Notebook 3 (construction filter, labels, context-only text, year split), so we explain exactly the test reports the classifier was scored on.

In [ ]:
csv_path = sorted(SIR_LOCAL.rglob("*.csv"), key=lambda p: p.stat().st_size)[-1]
raw = pd.read_csv(csv_path, encoding="latin-1", low_memory=False)
print(csv_path.name, raw.shape)

def find_col(df, *keys):
    # Return the first column whose simplified name contains all the given key words.
    simple = {c: re.sub(r"[^a-z]", "", c.lower()) for c in df.columns}
    for c, s in simple.items():
        if all(k in s for k in keys):
            return c
    raise KeyError(f"No column with {keys}. Columns: {list(df.columns)}")

COL = dict(date=find_col(raw, "eventdate"), naics=find_col(raw, "naics"), narrative=find_col(raw, "narrative"),
           event=find_col(raw, "event") if "Event" not in raw.columns else "Event",
           event_title=find_col(raw, "eventtitle"),
           body=find_col(raw, "partofbody") if "Part of Body" not in raw.columns else "Part of Body",
           body_title=find_col(raw, "partofbodytitle"), hosp=find_col(raw, "hospitalized"),
           amp=find_col(raw, "amputation"))
if any("eye" in c.lower() for c in raw.columns):        # 'Loss of Eye' exists in newer releases only
    COL["eye"] = find_col(raw, "eye")
else:
    raw["_no_eye"] = 0; COL["eye"] = "_no_eye"
print(json.dumps(COL, indent=1))

df = raw[[*COL.values()]].copy()
df.columns = list(COL.keys())
df = df[df.naics.astype(str).str.startswith("23")]
df = df.dropna(subset=["narrative"])
df["narrative"] = df.narrative.astype(str).str.strip()
df = df[df.narrative.str.len() > 20].drop_duplicates(subset=["narrative"])
df["year"] = pd.to_datetime(df.date, errors="coerce").dt.year
df = df.dropna(subset=["year"]); df["year"] = df.year.astype(int)
print("Construction reports with a narrative:", len(df), "| years", df.year.min(), "-", df.year.max())

In [ ]:
EVENT_GROUPS = {"1": "Violence by persons or animals", "2": "Transportation incident", "3": "Fire or explosion",
                "4": "Fall, slip or trip", "5": "Exposure to harmful substance or environment",
                "6": "Contact with objects or equipment", "7": "Overexertion or bodily reaction", "9": "Non-classifiable"}
BODY_GROUPS = {"1": "Head", "2": "Neck", "3": "Trunk", "4": "Upper limbs", "5": "Lower limbs",
               "6": "Body systems", "8": "Multiple body parts", "9": "Other or unspecified"}

def first_digit(x):
    # First digit of an OIICS code stored as number or text ('' if missing).
    s = re.sub(r"\D", "", str(x).split(".")[0])
    return s[0] if s else ""

df["event_group"] = df.event.map(first_digit).map(EVENT_GROUPS)
df["body_group"] = df.body.map(first_digit).map(BODY_GROUPS)
df["severity"] = ((pd.to_numeric(df.amp, errors="coerce").fillna(0) > 0) |
                  (pd.to_numeric(df.eye, errors="coerce").fillna(0) > 0)).map({True: "severe (amputation/eye loss)", False: "hospitalised"})
df = df.dropna(subset=["event_group", "body_group"])

for g in sorted(df.body_group.unique()):
    print(f"{g:22s} e.g. {list(df[df.body_group == g].body_title.value_counts().index[:3])}")

TARGETS = {"event_group": sorted(df.event_group.unique()),
           "body_group": sorted(df.body_group.unique()),
           "severity": sorted(df.severity.unique())}
label_counts = pd.concat({t: df[t].value_counts() for t in TARGETS}, axis=0).rename("reports")
label_counts.to_csv(RES / "osha_label_counts.csv")
print(label_counts)

In [ ]:
OUT_RE = re.compile(rf"\b({OUTCOME_WORDS})\b", re.I)
MECH_RE = re.compile(rf"\b({MECHANISM_WORDS})\b", re.I)

def clean_spaces(t):
    return re.sub(r"\s+", " ", re.sub(r"\s+([,.;])", r"\1", t)).strip()

def outcome_masked(t):
    # Remove words that reveal the injury or outcome.
    return clean_spaces(OUT_RE.sub(" ", t))

def context_only(t):
    # Remove outcome words AND words that say how the accident happened.
    return clean_spaces(MECH_RE.sub(" ", OUT_RE.sub(" ", t)))

df["text_full"] = df.narrative
df["text_outcome_masked"] = df.narrative.map(outcome_masked)
df["text_context_only"] = df.narrative.map(context_only)
TEXT_VERSIONS = {"full": "text_full", "outcome-masked": "text_outcome_masked", "context-only": "text_context_only"}

examples = df.sample(5, random_state=SEED)[["text_full", "text_outcome_masked", "text_context_only"]]
for _, r in examples.iterrows():
    print("FULL   :", r.text_full[:220]); print("OUTCOME:", r.text_outcome_masked[:220]); print("CONTEXT:", r.text_context_only[:220]); print("-" * 80)

In [ ]:
train_df = df[df.year <= TRAIN_MAX_YEAR].reset_index(drop=True)
val_df = df[df.year == VAL_YEAR].reset_index(drop=True)
test_df = df[df.year > VAL_YEAR].reset_index(drop=True)
split_sizes = {"train": len(train_df), "val": len(val_df), "test": len(test_df)}
print(split_sizes)

def oversample(d, col="event_group", minimum=OVERSAMPLE_MIN, seed=SEED):
    # Duplicate rows of rare classes (random oversampling) until each class has at least `minimum` rows.
    parts = []
    for _, g in d.groupby(col):
        parts.append(g)
        if len(g) < minimum:
            parts.append(g.sample(minimum - len(g), replace=True, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

train_os = oversample(train_df)
print("Training rows after oversampling:", len(train_os))
# Table: number of reports per accident type in each split (for the report's dataset section)
all_splits = pd.concat([train_df.assign(split="train"), val_df.assign(split="val"), test_df.assign(split="test")], ignore_index=True)
split_table = pd.crosstab(all_splits.event_group, all_splits.split)[["train", "val", "test"]]
split_table.to_csv(RES / "osha_split_by_event.csv"); print(split_table)

## 7. Choose the reports to explain
Random test reports (fixed seed) with at least `MIN_WORDS` words of context-only text. We explain the class the model **predicts**, because an explanation describes the model, right or wrong.

In [ ]:
pool = test_df[test_df.text_context_only.str.split().str.len() >= MIN_WORDS].sample(frac=1, random_state=SEED).reset_index(drop=True)
pool["doc_id"] = range(len(pool))
eval_docs = pool.head(N_EVAL)
global_docs = pool.head(max(N_GLOBAL, N_EVAL))
print("test reports:", len(test_df), "| usable:", len(pool), "| E10a-c:", len(eval_docs), "| E10d:", len(global_docs))

## 8. Experiment E10a-c · Explain every report with every method

For each report and method: word scores, the time taken, and the faithfulness tests. A **random** word ranking is scored the same way. Results are saved line by line in `xai_progress.jsonl` (and to your HF repo every 25 reports), so a stopped run continues.

In [ ]:
import numpy as np
done = {}
if PROGRESS.exists():
    for line in open(PROGRESS):
        r = json.loads(line); done[(r["doc_id"], r["task"], r["method"])] = r

def save_progress():
    if HF_REPO:
        try:
            HfApi(token=token).upload_file(path_or_fileobj=str(PROGRESS), path_in_repo="nb6_progress/xai_progress.jsonl", repo_id=HF_REPO)
        except Exception as e:
            print("progress upload failed:", e)

def run_one(doc, task, method):
    key = (int(doc.doc_id), task, method)
    if key in done:
        return done[key]
    text = doc.text_context_only
    if method == "random":
        n = len(text.split()); rng = np.random.default_rng(SEED + int(doc.doc_id))
        tgt = XP.predict_index(text, task)
        e = dict(words=text.split(), scores=rng.normal(size=n).tolist(), target=tgt, label=clf.targets[task][tgt],
                 prob=float(clf.proba([text], task)[0, tgt]), method="random", ms=0.0)
    else:
        e = XP.explain(text, task=task, method=method)
    e.update(faithfulness(XP.prob_fn(task, e["target"]), e["words"], e["scores"], ks=KS))
    e.update(doc_id=int(doc.doc_id), task=task, true=str(doc[task]), correct=bool(e["label"] == str(doc[task])))
    with open(PROGRESS, "a") as f:
        f.write(json.dumps(e) + "\n")
    done[key] = e
    return e

t_start = time.time()
for task in TASKS:
    for j, (_, doc) in enumerate(global_docs.iterrows()):
        for m in (METHODS + ["random"] if j < N_EVAL else ["shap"]):     # extra reports: SHAP only (for E10d)
            run_one(doc, task, m)
        if (j + 1) % 25 == 0:
            save_progress(); print(f"{task}: {j + 1}/{len(global_docs)} reports, {time.time() - t_start:.0f} s")
save_progress()
rows = pd.DataFrame([r for r in done.values() if r["doc_id"] < N_EVAL])
print(rows.groupby(["task", "method"]).size())

### E10a · Faithfulness table and the winner

**Faithfulness score** = mean over k of (comprehensiveness - sufficiency): high when the top words matter *and* are enough. The Wilcoxon signed-rank test checks if the winner is better than the runner-up on the same reports (paired test).

In [ ]:
from scipy.stats import wilcoxon
comp_cols, suff_cols = [f"comp@{k}" for k in KS], [f"suff@{k}" for k in KS]
rows["faithfulness"] = rows[comp_cols].mean(axis=1) - rows[suff_cols].mean(axis=1)
summary = rows.groupby(["task", "method"])[comp_cols + suff_cols + ["faithfulness", "ms"]].mean()
print(summary.round(4).to_string())

choice = {}
for task in TASKS:
    s = summary.loc[task].drop("random")["faithfulness"].sort_values(ascending=False)
    best, second = s.index[0], s.index[1]
    a = rows[(rows.task == task) & (rows.method == best)].sort_values("doc_id").faithfulness.values
    b = rows[(rows.task == task) & (rows.method == second)].sort_values("doc_id").faithfulness.values
    r = rows[(rows.task == task) & (rows.method == "random")].sort_values("doc_id").faithfulness.values
    p_second = wilcoxon(a, b).pvalue if len(a) > 5 and np.any(a != b) else float("nan")
    p_random = wilcoxon(a, r).pvalue if len(a) > 5 and np.any(a != r) else float("nan")
    choice[task] = dict(best_method=best, runner_up=second, p_vs_runner_up=float(p_second), p_vs_random=float(p_random))
    for m, row in summary.loc[task].iterrows():
        for metric, v in row.items():
            log_result("E10", f"E10a_{m}", m, task, metric, v)
    log_result("E10", "E10a_test", best, task, "wilcoxon_p_vs_runner_up", p_second, f"runner-up {second}")
    log_result("E10", "E10a_test", best, task, "wilcoxon_p_vs_random", p_random)
    print(f"{task}: best = {best.upper()} (vs {second}: p = {p_second:.3g}; vs random: p = {p_random:.3g})")

main = choice[TASKS[0]]
json.dump({"best_method": main["best_method"], "per_task": choice, "ks": list(KS),
           "settings": dict(shap_evals=SHAP_EVALS, lime_samples=LIME_SAMPLES, ig_steps=IG_STEPS, n_eval=len(eval_docs))},
          open(RES / "xai_choice.json", "w"), indent=1)
summary.to_csv(RES / "E10a_faithfulness.csv")

In [ ]:
import matplotlib.pyplot as plt
task = TASKS[0]; order = METHODS + ["random"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for m in order:
    ax[0].plot(KS, [summary.loc[(task, m), f"comp@{k}"] for k in KS], marker="o", label=m.upper())
    ax[1].plot(KS, [summary.loc[(task, m), f"suff@{k}"] for k in KS], marker="o", label=m.upper())
ax[0].set(title="Comprehensiveness (higher = better)", xlabel="top-k words removed", ylabel="drop in confidence")
ax[1].set(title="Sufficiency (lower = better)", xlabel="top-k words kept", ylabel="drop in confidence")
for a in ax: a.grid(alpha=.3); a.legend(); a.set_xticks(list(KS))
plt.suptitle(f"E10a · Faithfulness of word explanations ({len(eval_docs)} OSHA test reports)")
plt.tight_layout(); plt.savefig(FIGS / "E10a_faithfulness.png", dpi=150); plt.show()

### E10b · Do the methods agree? · E10c · Speed
Spearman correlation of the word scores (1 = same ranking) and overlap of the top-3 words (Jaccard, 1 = same 3 words), averaged over reports. Speed is the mean time per explanation on this GPU.

In [ ]:
from scipy.stats import spearmanr
from itertools import combinations
agree = []
for task in TASKS:
    by = {m: rows[(rows.task == task) & (rows.method == m)].set_index("doc_id") for m in METHODS}
    for a, b in combinations(METHODS, 2):
        rho, jac = [], []
        for d in by[a].index.intersection(by[b].index):
            sa, sb = by[a].loc[d, "scores"], by[b].loc[d, "scores"]
            if len(sa) > 2 and np.std(sa) > 0 and np.std(sb) > 0:
                rho.append(spearmanr(sa, sb).correlation)
            ta, tb = top_k_positions(sa, 3), top_k_positions(sb, 3)
            jac.append(len(ta & tb) / len(ta | tb))
        agree.append(dict(task=task, pair=f"{a}-{b}", spearman=np.nanmean(rho), top3_jaccard=np.mean(jac)))
        log_result("E10", f"E10b_{a}_{b}", f"{a}-{b}", task, "spearman", np.nanmean(rho))
        log_result("E10", f"E10b_{a}_{b}", f"{a}-{b}", task, "top3_jaccard", np.mean(jac))
agree = pd.DataFrame(agree); print(agree.round(3))
speed = rows[rows.method != "random"].groupby("method").ms.describe()[["mean", "50%", "max"]].round(1)
for m, r in speed.iterrows():
    log_result("E10", f"E10c_{m}", m, TASKS[0], "ms_per_explanation", r["mean"], DEVICE)
print(speed)
agree.to_csv(RES / "E10b_agreement.csv", index=False); speed.to_csv(RES / "E10c_speed.csv")

## 9. Experiment E10d · What does the model rely on overall? (global word importance)

SHAP values of many reports are averaged per word and per predicted accident type (Lundberg et al. [X1] call this a global summary). Only positive contributions are counted, and a word must appear in at least `MIN_WORD_COUNT` reports. Common words (*the, and, employee*) and numbers are left out. This shows the **evidence words** the model uses for each accident type, e.g. *ladder, roof, scaffold* for falls.

In [ ]:
from collections import defaultdict
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS     # 'the', 'and', 'his'... are not useful evidence words
STOP = set(ENGLISH_STOP_WORDS) | {"employee", "employees", "worker", "workers", "approximately"}
task = TASKS[0]
sh = [r for r in done.values() if r["task"] == task and r["method"] == "shap"]
acc = defaultdict(lambda: defaultdict(list))
for r in sh:
    seen = set()
    for w, s in zip(r["words"], r["scores"]):
        w = re.sub(r"[^a-z0-9\-]", "", w.lower())
        if len(w) > 2 and w not in seen and w not in STOP and not w.isdigit():
            acc[r["label"]][w].append(max(s, 0)); seen.add(w)
glob = []
for label, words in acc.items():
    for w, v in words.items():
        if len(v) >= MIN_WORD_COUNT:
            glob.append(dict(accident_type=label, word=w, mean_shap=np.mean(v), reports=len(v)))
glob = pd.DataFrame(glob).sort_values(["accident_type", "mean_shap"], ascending=[True, False])
top = glob.groupby("accident_type").head(10)
top.to_csv(RES / "E10d_global_top_words.csv", index=False)
labels = [l for l in top.accident_type.unique()]
fig, axes = plt.subplots(int(np.ceil(len(labels) / 3)), 3, figsize=(14, 3.2 * int(np.ceil(len(labels) / 3))), squeeze=False)
for ax, l in zip(axes.flat, labels):
    d = top[top.accident_type == l].iloc[::-1]
    ax.barh(d.word, d.mean_shap, color="#2563eb"); ax.set_title(f"{l} (n={len([r for r in sh if r['label'] == l])})", fontsize=9)
for ax in list(axes.flat)[len(labels):]:
    ax.axis("off")
plt.suptitle("E10d · Words that push the model towards each accident type (mean SHAP)"); plt.tight_layout()
plt.savefig(FIGS / "E10d_global_words.png", dpi=150); plt.show()
print(top.groupby("accident_type").word.apply(lambda s: ", ".join(s)).to_string())

## 10. Experiment E10e · Examples, including SafeVision-style situations
Green words push towards the predicted accident type, red words push away. The last examples are sentences like the ones SafeVision builds from a photo.

In [ ]:
from IPython.display import HTML, display
best = main["best_method"]
APP_SITUATIONS = [
    "Construction site with 3 workers. 1 worker is not wearing a hard hat. Workers are on a scaffold near the roof edge.",
    "2 workers without high-visibility vests next to a reversing excavator and a dump truck on site.",
    "Worker without gloves cutting steel rebar with a portable grinder near the formwork.",
    "Worker without safety glasses drilling concrete overhead on a ladder.",
]
def colour(words, scores):
    m = max(1e-9, max(abs(s) for s in scores)) if scores else 1
    span = lambda w, s: (f'<span style="background:rgba({"22,163,74" if s > 0 else "220,38,38"},{abs(s) / m:.2f});'
                         f'padding:1px 3px;border-radius:3px">{w}</span>')
    return " ".join(span(w, s) for w, s in zip(words, scores))
examples, html = [], ""
for t in list(eval_docs.text_context_only.head(3)) + [clf.context_only(s) for s in APP_SITUATIONS]:
    e = XP.explain(t, task=TASKS[0], method=best)
    examples.append(e)
    html += f"<p><b>{e['label']}</b> ({e['prob']:.2f}, {best.upper()})<br>{colour(e['words'], e['scores'])}</p>"
display(HTML(html)); (FIGS / "E10e_examples.html").write_text(html)
json.dump(examples, open(RES / "E10e_examples.json", "w"), indent=1)

## 11. Save everything
Excel with all E10 results, `xai_choice.json` (copy it to `backend/models/` for the app), a ZIP of the outputs and an upload to your HF repo (`nb6_explainability`).

In [ ]:
import shutil
log = read_log()
with pd.ExcelWriter(RES / "E10_explainability.xlsx") as xw:
    log.to_excel(xw, "all_results", index=False)
    summary.reset_index().to_excel(xw, "E10a_faithfulness", index=False)
    agree.to_excel(xw, "E10b_agreement", index=False)
    speed.reset_index().to_excel(xw, "E10c_speed", index=False)
    top.to_excel(xw, "E10d_global_words", index=False)
archive = shutil.make_archive(str(OUT.parent / "safevision_nb6_outputs"), "zip", OUT)
print("Saved", archive, "| best method:", main["best_method"].upper())

In [ ]:
# [colab-only]
if HF_REPO:
    HfApi(token=token).upload_folder(folder_path=str(OUT), repo_id=HF_REPO, path_in_repo="nb6_explainability")
    print("Uploaded to", HF_REPO, "/ nb6_explainability")
if IN_COLAB:
    from google.colab import files
    files.download(archive)

## References
- [X1] S. M. Lundberg and S.-I. Lee, "A unified approach to interpreting model predictions," *NeurIPS*, 2017.
- [X2] M. T. Ribeiro, S. Singh and C. Guestrin, "'Why should I trust you?': Explaining the predictions of any classifier," *KDD*, 2016.
- [X3] M. Sundararajan, A. Taly and Q. Yan, "Axiomatic attribution for deep networks," *ICML*, 2017.
- [X4] J. DeYoung et al., "ERASER: A benchmark to evaluate rationalized NLP models," *ACL*, 2020.
- [6] Wang et al. 2025 and [7] Cao et al. 2026: the OSHA accident classifiers that Notebook 3 follows.